# 42 · Sample a raster at points or over areas, checked against rasterio and SciPy

## What this checks

**Sample a raster at points or over areas** reads a grid's value where each point is. This check covers release
0.10.1, as part of a whole chain of steps.

The chain is the sea-level example's third workflow, around Manila. It asks which health facilities stand on ground
below the sea level projected for 2100.

- **Raster input** keeps three published grids around Manila.
- **Sample a raster** reads them at the sea-level sites and at the facilities.
- **The workflow's own rules** combine the heights.
- **A published rule set**, compare-level-to-elevation 1.0.0, sets each facility against the rise at 2100 from AR6,
  the projections of the IPCC's sixth assessment.

It found 16 of 194 facilities on ground below that level. A wrong height could move a facility to the wrong side of
the line.

## How it is checked

We read the same three grids again, with tools that share no code with Fieldwork.

- **The grids.** **rasterio**, a Python reader for map grids built on GDAL, reads all three.
- **The sea-surface grids.** EGM2008 is a model of the geoid, the level the sea would take with no tides or currents.
  DTU25MSS is a model of the mean sea surface. Both are published as values at grid points. **SciPy** interpolates
  between them: it takes a weighted average of the four nearest points, called bilinear interpolation.
- **The ground grid.** DeltaDTM is a model of ground height along coasts. Its value is the cell rasterio picks.
- **The nearest site** is found in **NumPy**, by distance on a sphere (the haversine formula).
- **The heights and the comparison** are written out from the arithmetic the data declare.

The data are the chain as Fieldwork ran it, with the three grid windows the workflow sampled.

## What should happen

1. **At each sea-level site**, EGM2008 and DTU25MSS are within 10⁻⁹ m of SciPy's value. A site outside the window
   has no value on either side.
2. **At each facility**, the ground height is exactly rasterio's cell value, and the nearest site is the same.
3. **Every height**, sea above the geoid and ground elevation, is within 10⁻⁹ m of the declared arithmetic.
4. **Every level** is the same, Reaches or StaysBelow, and so is the count of facilities that Reach.

We also record, without judging, any facility within 1 mm of the projected level. There, rounding on either side
could tip it.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports: rasterio cannot share this kernel with pyproj, fiona or geopandas (check 00).
NEEDED = ["numpy", "rasterio", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_42_sea_level_heights import run

report = run(Path.cwd() / "fixtures" / "sea-level-heights-manila-001.json")
print("agrees:", report["agrees"])
print(f"Reaches: {report['reaches']} of {report['facilities']} here, {report['fieldworkReaches']} in Fieldwork")
m = report["manila"]
print(f"At Manila's site: EGM2008 {m['egm2008M']:.4f} m, DTU25MSS {m['mssM']:.4f} m, sea above the geoid {m['seaAboveGeoidM']:.4f} m")
for s in report["sites"]:
    print(("OK " if s["agrees"] else "XX ") + s["site_id"], "outside the window" if s["scipy"]["egm2008M"] is None else "")
print("differs:", report["differs"] or "nowhere")
print("within 1 mm of the projected level:", report["withinOneMillimetre"] or "none")

## The result on record

First recorded on 9 November 2026, against release 0.2.0. Rerun on 26 November 2026, against release 0.10.1:
**Agrees**.

- **The sea-level site.** EGM2008 is 43.0423 m and DTU25MSS is 43.9426 m, within 10⁻⁹ m of SciPy. Seven sites lie
  outside the window and have no value on either side.
- **The facilities.** For all 194, each ground height is exactly rasterio's cell, and each nearest site is the same.
  Every height is within 10⁻⁹ m.
- **The level.** 16 of 194 facilities Reach on both sides. None is within 1 mm of the line.

The result agrees in CPython and in the browser kernel. Building the same workflow by hand on the canvas gives the
same 16.

Running the cell above repeats the check on the same data.

## What this does not show

- That a facility below the projected level will flood. There is no tide, storm surge, sea defence or drainage
  here.
- That the ground heights are exact at the coast. DeltaDTM is least certain there; its stated error is about 0.45 m.
- That the EGM2008 window is the same as PROJ's whole grid. Its value at the gauge, 43.009 m, is checked against
  pyproj in Fieldwork's own tests. pyproj cannot run in the same kernel as rasterio.
- That OpenStreetMap's 194 facilities are all there are.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
e = report["environment"]
print("rasterio:", e["rasterio"], "· GDAL", e["gdal"], "· SciPy", e["scipy"], "· NumPy", e["numpy"])
print("runtime :", e["runtime"])